# 03 - Player Features: Multi-Accounting Group

Purpose: build and verify only the Phase 3 `ma_` feature group from the frozen snapshot.

Input: `frauddet.snapshot.load_snapshot` using `config.yaml` `phase3.input_dir`. No live Mongo and no mutable top-level parquet inputs.

Outputs:
- `data/player_features.parquet`: one row per canonical player, wide scalar + metadata columns.
- `data/player_features_evidence.parquet`: one row per `player_key + feature_name`; `feature_evidence` is JSON reviewer evidence.

Evidence uses direct one-hop links only. Identity hashes remain hashed; no raw NIN/email is present. Tier 4 IP/bank/passport features are deferred.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from frauddet import config
from frauddet.features.multi_accounting import (
    FEATURE_SPECS,
    build_multi_account_linkages,
    build_multi_accounting_features,
)
from frauddet.snapshot import load_snapshot, snapshot_dir

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)
print("snapshot_dir", snapshot_dir())

snapshot_dir C:\Users\dev\OneDrive\Documents\Fraud Detection - FS Book\Fraud-Detection-FS-Book\data\snapshot_phase3_2026-06-19b


## Build

In [2]:
result = build_multi_accounting_features()
player_features = result.player_features
feature_evidence = result.feature_evidence
result.output_paths, player_features.shape, feature_evidence.shape

({'player_features': WindowsPath('C:/Users/dev/OneDrive/Documents/Fraud Detection - FS Book/Fraud-Detection-FS-Book/data/player_features.parquet'),
  'feature_evidence': WindowsPath('C:/Users/dev/OneDrive/Documents/Fraud Detection - FS Book/Fraud-Detection-FS-Book/data/player_features_evidence.parquet')},
 (116, 37),
 (1044, 3))

## Feature Coverage

In [3]:
coverage_rows = []
for feature_name, spec in FEATURE_SPECS.items():
    values = player_features[feature_name]
    coverage_rows.append(
        {
            "feature_name": feature_name,
            "strength": spec.strength,
            "scoring_role": spec.scoring_role,
            "players": len(values),
            "non_null": int(values.notna().sum()),
            "non_zero": int(values.fillna(0).astype(float).ne(0).sum()),
            "null": int(values.isna().sum()),
        }
    )
coverage = pd.DataFrame(coverage_rows)
display(coverage)

assert int(coverage.loc[coverage.feature_name.eq("ma_email_shared_account_count"), "non_zero"].iloc[0]) == 2
assert int(coverage.loc[coverage.feature_name.eq("ma_nin_shared_account_count"), "non_zero"].iloc[0]) == 0
assert int(coverage.loc[coverage.feature_name.eq("ma_identity_phone_collision_count"), "non_zero"].iloc[0]) == 0
assert player_features["ma_nin_shared_account_count"].notna().all()
assert player_features["ma_identity_phone_collision_count"].notna().all()

# Device-link contamination guard: measure all valid fingerprints, but link only
# fingerprints whose population cardinality is at or below the configured cap.
players_raw = load_snapshot("players")
logins_raw = load_snapshot("logins")
money_raw = load_snapshot("money")
linkages = build_multi_account_linkages(players_raw, logins_raw, money_raw)
device_cap = int(config.load_config()["thresholds"]["ma_device_max_cardinality"])
excluded_device_cardinalities = sorted(
    [
        len(player_keys)
        for player_keys in linkages.device_all.key_to_players.values()
        if len(player_keys) > device_cap
    ],
    reverse=True,
)
print("device_max_cardinality", device_cap)
print("excluded_fingerprint_count", len(excluded_device_cardinalities))
print("excluded_fingerprint_cardinalities", excluded_device_cardinalities)
assert all(
    len(player_keys) <= device_cap
    for player_keys in linkages.device.key_to_players.values()
)

,feature_name,strength,scoring_role,players,non_null,non_zero,null
0,ma_nin_shared_account_count,strong,scoring,116,116,0,0
1,ma_email_shared_account_count,strong,scoring,116,116,2,0
2,ma_device_shared_account_count,strong,scoring,116,87,43,29
3,ma_withdrawal_recipient_shared_count,strong,scoring,116,10,0,106
4,ma_identity_phone_collision_count,strong,scoring,116,116,0,0
5,ma_referred_by_linked_account,strong,supporting,116,116,0,0
6,ma_referral_fanout_count,weak,supporting,116,116,2,0
7,ma_cocreated_linked_count,moderate,supporting,116,116,9,0
8,ma_device_count,weak,context_only,116,87,87,29


device_max_cardinality 10
excluded_fingerprint_count 5
excluded_fingerprint_cardinalities [28, 18, 15, 11, 11]


## Evidence Integrity

In [4]:
evidence_lookup = {
    (row.player_key, row.feature_name): json.loads(row.feature_evidence)
    for row in feature_evidence.itertuples(index=False)
}
for row in player_features.itertuples(index=False):
    values = row._asdict()
    for feature_name, spec in FEATURE_SPECS.items():
        value = values[feature_name]
        if spec.scoring_role == "scoring" and pd.notna(value) and bool(value):
            assert evidence_lookup[(values["player_key"], feature_name)]
print("VERIFIED: every non-zero scoring ma_ feature has non-empty evidence")

excluded_fingerprints = {
    shared_key
    for shared_key, player_keys in linkages.device_all.key_to_players.items()
    if len(player_keys) > device_cap
}
for row in feature_evidence.loc[
    feature_evidence.feature_name.isin(
        {
            "ma_device_shared_account_count",
            "ma_referred_by_linked_account",
            "ma_cocreated_linked_count",
        }
    )
].itertuples(index=False):
    assert not any(
        fingerprint in row.feature_evidence
        for fingerprint in excluded_fingerprints
    )
print("VERIFIED: over-cardinality fingerprints are absent from link/corroboration evidence")

VERIFIED: every non-zero scoring ma_ feature has non-empty evidence
VERIFIED: over-cardinality fingerprints are absent from link/corroboration evidence


## Hand Reconciliation Against Frozen Parquet

In [5]:
players_raw = load_snapshot("players")
logins_raw = load_snapshot("logins")
money_raw = load_snapshot("money")

email_players = player_features.loc[
    player_features["ma_email_shared_account_count"].gt(0), "player_key"
].astype(str).tolist()
referral_players = player_features.loc[
    player_features["ma_referred_by_linked_account"].fillna(False), "player_key"
].astype(str).tolist()
device_players = player_features.loc[
    player_features["ma_device_shared_account_count"].fillna(0).gt(0), "player_key"
].astype(str).tolist()
picked_players = list(
    dict.fromkeys(email_players + referral_players[:1] + device_players[:1])
)[:3]

print("picked_players", picked_players)
display(
    player_features.loc[
        player_features.player_key.isin(picked_players),
        ["player_key"] + list(FEATURE_SPECS),
    ]
)

focus_features = {
    "ma_email_shared_account_count",
    "ma_referred_by_linked_account",
    "ma_device_shared_account_count",
}
focus_evidence = feature_evidence[
    feature_evidence.player_key.isin(picked_players)
    & feature_evidence.feature_name.isin(focus_features)
].copy()
focus_evidence["parsed_evidence"] = focus_evidence.feature_evidence.map(json.loads)
display(focus_evidence[["player_key", "feature_name", "parsed_evidence"]])

linked_players = set(picked_players)
for evidence in focus_evidence.feature_evidence.map(json.loads):
    for item in evidence:
        linked_players.update(item.get("other_player_keys", []))
        if item.get("referrer_player_key"):
            linked_players.add(item["referrer_player_key"])

# Raw frozen rows needed to reconcile the selected hashes/referrals/devices.
display(
    players_raw.loc[
        players_raw.player_key.isin(linked_players),
        ["player_key", "created_at", "nin_hash", "email_hash", "referred_by_key"],
    ].sort_values("player_key")
)

shared_fingerprints = set()
for evidence in focus_evidence.feature_evidence.map(json.loads):
    for item in evidence:
        if item.get("shared_key_type") == "fingerprint":
            shared_fingerprints.add(item["shared_key"])
        for linkage in item.get("corroborating_linkages", []):
            if linkage.get("shared_key_type") == "fingerprint":
                shared_fingerprints.add(linkage["shared_key"])

display(
    logins_raw.loc[
        logins_raw.player_key.isin(linked_players)
        & logins_raw.fingerprint.isin(shared_fingerprints),
        ["player_key", "source_id", "fingerprint", "created_at"],
    ].drop_duplicates().sort_values(["fingerprint", "player_key", "created_at"])
)

# No shared-recipient feature fires in this snapshot, but show picked players' completed withdrawals if any.
display(
    money_raw.loc[
        money_raw.player_key.isin(picked_players)
        & money_raw.txn_type.eq("WITHDRAWAL")
        & money_raw.is_money_out.fillna(False),
        ["player_key", "transaction_id", "recipient_normalized", "requested_at"],
    ].sort_values(["player_key", "requested_at"])
)

picked_players ['6a26959dfc099dd782863ecf', '6a311326aa7ed994dfda83af', '6a0ea9ff174ad3c431d9e16d']


,player_key,ma_nin_shared_account_count,ma_email_shared_account_count,ma_device_shared_account_count,ma_withdrawal_recipient_shared_count,ma_identity_phone_collision_count,ma_referred_by_linked_account,ma_referral_fanout_count,ma_cocreated_linked_count,ma_device_count
0,6a0ea9ff174ad3c431d9e16d,0,0,9,0,0,False,0,1,8
41,6a26959dfc099dd782863ecf,0,1,0,<NA>,0,False,0,0,2
84,6a311326aa7ed994dfda83af,0,1,2,0,0,False,0,0,2


,player_key,feature_name,parsed_evidence
2,6a0ea9ff174ad3c431d9e16d,ma_device_shared_account_count,"[{'linked_source_record_ids': {'6a0eabae174ad3c431d9e6b6': ['6a0f1dc0a5d7ce78c657bf45'], '6a0f0df7a5d7ce78c657a62f': ['6a0f0e07a5d7ce78c657a649', '6a0f1e89a..."
3,6a0ea9ff174ad3c431d9e16d,ma_email_shared_account_count,[]
7,6a0ea9ff174ad3c431d9e16d,ma_referred_by_linked_account,[]
371,6a26959dfc099dd782863ecf,ma_device_shared_account_count,[]
372,6a26959dfc099dd782863ecf,ma_email_shared_account_count,"[{'linked_source_record_ids': {'6a311326aa7ed994dfda83af': []}, 'other_player_keys': ['6a311326aa7ed994dfda83af'], 'shared_key': '8e922392304ca6feb6fc074475..."
376,6a26959dfc099dd782863ecf,ma_referred_by_linked_account,[]
758,6a311326aa7ed994dfda83af,ma_device_shared_account_count,"[{'linked_source_record_ids': {'6a0eac61174ad3c431d9e947': ['6a1041b3a5d7ce78c65809f5', '6a106245a5d7ce78c65819a8', '6a11d71c0a5fadca63729865', '6a140b850a5..."
759,6a311326aa7ed994dfda83af,ma_email_shared_account_count,"[{'linked_source_record_ids': {'6a26959dfc099dd782863ecf': []}, 'other_player_keys': ['6a26959dfc099dd782863ecf'], 'shared_key': '8e922392304ca6feb6fc074475..."
763,6a311326aa7ed994dfda83af,ma_referred_by_linked_account,[]


,player_key,created_at,nin_hash,email_hash,referred_by_key
0,6a0ea9ff174ad3c431d9e16d,2026-05-21 06:45:19.652000+00:00,475cd04fd275c003f8b8a2393132d6076582ef46f17dff8ac4ad0ab5b3790965,c2f8254bfd7b70b9729b963ae7883886273a5c739bc682dd079f5543e9ca24a1,NaN
1,6a0eabae174ad3c431d9e6b6,2026-05-21 06:52:30.870000+00:00,NaN,1a29e014b69b8f8e8d31aae3b6b7544b7fd7f0150ce2931e9ab1363c1bd77d9e,NaN
2,6a0eac61174ad3c431d9e947,2026-05-21 06:55:29.137000+00:00,NaN,9ee553e33a1add77b8a3b84e62e8a4fbfb5227c16a7dc739fb3b43a667b83dcf,NaN
6,6a0eaefe9d615bad27c7f786,2026-05-21 07:06:39.115000+00:00,NaN,799cac9a1358fc863496a205c64a5ccda1a869fd7ace70ffc299cbe7662d7e70,NaN
7,6a0ef9cca5d7ce78c65765d9,2026-05-21 12:25:48.725000+00:00,NaN,ee5fa051aabbad3d0458007a4be7246232cdff2cb260811e0038381518c7ffdb,NaN
8,6a0f0df7a5d7ce78c657a62f,2026-05-21 13:51:51.182000+00:00,NaN,3f6d4dccfe89d1ac131b4a3f3d34cb5fd693ba89050e73111567c1993927d598,6a0ef9cca5d7ce78c65765d9
28,6a2161017364bea0f4f5aede,2026-06-04 11:26:57.619000+00:00,1e051f637a49a1680ec9d238412700c294d6488a376b7fd4c1923d4dd432de9b,NaN,NaN
29,6a21621d7364bea0f4f5b21f,2026-06-04 11:31:41.764000+00:00,474187d39d09a2f4a2c4a130500321c06fb813b7161a7fc13c168cbbbc10a3ac,NaN,NaN
30,6a21639b7364bea0f4f5b356,2026-06-04 11:38:03.863000+00:00,7fd091345fa7dd2973f9e8fc55b5dc89f048f5352ef94fc48b2a90fa299c5159,NaN,NaN
37,6a22bb07ceed05a02d03c8c7,2026-06-05 12:03:19.043000+00:00,dea4f18ac53d2036507a8f5d72eac12f8c284f2327f18f880920aa4a917b3b3a,NaN,NaN


,player_key,source_id,fingerprint,created_at
60,6a0ea9ff174ad3c431d9e16d,6a0f0241a5d7ce78c65781b8,1a42e272f81931ac002de1196d54742e0fce3a1cff42b79a9a5aad42047e0167,2026-05-21 13:01:53.275000+00:00
114,6a0ea9ff174ad3c431d9e16d,6a104c4da5d7ce78c6580f7f,1a42e272f81931ac002de1196d54742e0fce3a1cff42b79a9a5aad42047e0167,2026-05-22 12:30:05.340000+00:00
137,6a0ea9ff174ad3c431d9e16d,6a11fbfb0a5fadca6372aab7,1a42e272f81931ac002de1196d54742e0fce3a1cff42b79a9a5aad42047e0167,2026-05-23 19:11:55.370000+00:00
144,6a0ea9ff174ad3c431d9e16d,6a13359f0a5fadca6372bbd8,1a42e272f81931ac002de1196d54742e0fce3a1cff42b79a9a5aad42047e0167,2026-05-24 17:30:07.574000+00:00
70,6a0eabae174ad3c431d9e6b6,6a0f1dc0a5d7ce78c657bf45,1a42e272f81931ac002de1196d54742e0fce3a1cff42b79a9a5aad42047e0167,2026-05-21 14:59:12.087000+00:00
...,...,...,...,...
284,6a2161017364bea0f4f5aede,6a1c265ca749eff07ed9315e,d922849ec0d12e5598ddcc188d24b1e4b6ed6be07c0817fb50ec5affbf76eb68,2026-05-31 12:15:24.281000+00:00
277,6a22bb07ceed05a02d03c8c7,6a1c0874a749eff07ed924ce,d922849ec0d12e5598ddcc188d24b1e4b6ed6be07c0817fb50ec5affbf76eb68,2026-05-31 10:07:48.878000+00:00
287,6a22bb07ceed05a02d03c8c7,6a1c27f8a749eff07ed932bf,d922849ec0d12e5598ddcc188d24b1e4b6ed6be07c0817fb50ec5affbf76eb68,2026-05-31 12:22:16.829000+00:00
288,6a22bb07ceed05a02d03c8c7,6a1c2849a749eff07ed9331e,d922849ec0d12e5598ddcc188d24b1e4b6ed6be07c0817fb50ec5affbf76eb68,2026-05-31 12:23:37.243000+00:00


,player_key,transaction_id,recipient_normalized,requested_at
43,6a0ea9ff174ad3c431d9e16d,TXNY_MPWO99LH_81C8EA28AE08,757575757,2026-06-02 13:27:09.384000+00:00
44,6a0ea9ff174ad3c431d9e16d,TXNY_MPWOA1IW_1CE1BC90EF45,757575757,2026-06-02 13:27:45.549000+00:00
1138,6a311326aa7ed994dfda83af,TXNY_MQGIA50Q_275478271E03,755432100,2026-06-16 10:35:15.894000+00:00


## FINDINGS — Multi-accounting features — frozen Phase 3 snapshot — 2026-06-22
- Verified: `player_features.parquet` has one row for each of 116 canonical snapshot players and contains only the nine requested Tier 1-3 `ma_` features plus per-feature null/strength/role metadata.
- Verified: evidence store schema is long-form `(player_key, feature_name, feature_evidence)` with JSON evidence. It contains 1,044 rows (116 players x 9 features); non-zero scoring features always have evidence, while zero/null values may honestly carry empty evidence.
- Verified: device sharing uses `ma_device_max_cardinality=10`. Five office/public fingerprints were excluded with player cardinalities `[28, 18, 15, 11, 11]`; excluded hashes are absent from device-link, referral-corroboration, and co-creation evidence.
- Verified: `ma_nin_shared_account_count` is non-null for 116/116 and non-zero for 0; absent NIN correctly maps to measured `0`, not null.
- Verified: `ma_email_shared_account_count` is non-null for 116/116 and non-zero for 2, representing the single frozen shared-email pair.
- Verified: `ma_device_shared_account_count` is measurable for 87/116 and non-zero for 43 after the cap (previously 85); 29 players remain null with `no_valid_fingerprint_logins`.
- Verified: `ma_withdrawal_recipient_shared_count` is measurable for 10/116 and non-zero for 0; 106 players are null with `no_completed_withdrawals`.
- Verified: `ma_identity_phone_collision_count` is non-null for 116/116 and non-zero for 0, confirming the dormant phone-collision path is wired.
- Verified: `ma_referred_by_linked_account` is non-null for 116/116 and non-zero for 0 after the cap (previously 3); the previous hits were entirely office-fingerprint contamination.
- Verified: `ma_referral_fanout_count` is non-null for 116/116 and non-zero for 2.
- Verified: `ma_cocreated_linked_count` is non-null for 116/116 and non-zero for 9 after the cap (previously 29), using the configured 15-minute window plus a direct non-excluded shared signal.
- Verified: `ma_device_count` remains measurable/non-zero for 87/116 because it counts all valid observed fingerprints, including public/office devices for context; 29 are null with `no_valid_fingerprint_logins`.
- Surprises: Capping five high-cardinality fingerprints removed every referral-linked hit and two-thirds of co-created-linked hits, confirming the uncapped office fingerprints were contaminating stronger features.
- Gaps: Tier 4 IP/bank/passport features remain documented TODOs only. The device cap is a placeholder and must be recalibrated on production data.
- Decisions needed: None for this hardening pass. Review before proceeding to `pay_`.
- Updated assumptions: high-cardinality fingerprints remain visible to `ma_device_count` as context but cannot create device links or corroborate referral/co-creation features.